# Bring Your Own Data (BYOD)

In this guide, we’ll walk you through how to integrate your own dataset into a LEIP Design recipe. We will start by reviewing the datasets we offer, followed by a step-by-step demonstration using the Road Sign Detection dataset from Kaggle. These steps can be applied to any dataset you choose to work with.

This guide specifically focuses on **object detection** and is designed for integrating **detection datasets** into a LEIP Design recipe.

First, we generate our pantry and create a recipe to work with, as shown in the [Getting Started tutorial](https://docs.latentai.io/leip/design/latest/notebooks/GettingStartedwithLEIPDesign/).

In [ ]:
from pathlib import Path
import leip_recipe_designer as rd

# Define the workspace path
workspace = Path('./workspace')

# Build the pantry (do not rebuild if it already exists)
pantry = rd.Pantry.build(workspace / "./my_combined_pantry/", force_rebuild=False)
recipe = rd.create.from_recipe_id('44702', pantry=pantry, allow_upgrade=True)

Before bringing your own dataset, you might want to explore the object detection datasets we provide. These can be a good starting point if you’re looking for a quick test setup.

In [2]:
recipe.options("data_generator")

> Help: Dataset. 
> Ingredients that fit:
  Index  Parameter                                                                                       Type                                Version    UUID
      0  BYOD from Url - PASCAL format                                                                   data_generator.vision.detection.2d  1.0.0      da9ec6daa3a287173c17307eb727a033c356677976662fca84c1d0697c5960ef
      1  BYOD from Url - COCO format                                                                     data_generator.vision.detection.2d  1.0.0      0b9bca30d0fe77ee7734e56dc7291272c7e69894d3419ac64f70fe32add19f32
      2  BYOD from Url - YOLO format                                                                     data_generator.vision.detection.2d  1.0.0      104f9721404c4653f2eddbe04023b4858a72aff3cec2937e05c714ac1d2d91c8
      3  BYOD from Url - KITTI format                                                                    data_generator.vision.detection.2d  1.0.0    

**Assigning Ingredients to Your Recipe:**

To assign a data generator to your recipe, use the `assign_ingredients` method. This approach is recommended when building a recipe from scratch using one of our provided datasets.

In [3]:
recipe.assign_ingredients('data_generator', "COCO Car Detection")

[{'choice_id': 'a7c5c41ec12a9c70f6b8264226968b7498bc9b5d2df8914b0aa5518fd3b661c1',
  'choice_name': 'COCO Car Detection (data/sets/kaggle/detection/coco-car-dataset)',
  'synonym': 'data_generator',
  'parent': 'Basic Adaptor',
  'slot': 'slot:module.dataset_generator',
  'path': ['slot:data', 'slot:module.dataset_generator']}]

For more details on initializing an empty recipe for your tasks, refer to the [Recipe Creators documentation](http://docs.latentai.io/leip/design/latest/content/reference/recipe_creators/).

> **Note:** The `assign_ingredients` function is best used when creating a new recipe, as it clears and initializes preprocessing steps such as augmentations from scratch.

If you are modifying one of our pre-validated "golden recipes" and wish to retain advanced augmentations like mosaicing that contribute to optimal performance, use the `replace_data_generator` method instead:


In [4]:
recipe = rd.create.from_recipe_id('44702', pantry=pantry, allow_upgrade=True)
data = rd.helpers.data.get_data_generator_by_name(pantry=pantry, regex_ingredient_name="COCO Car Detection")
rd.helpers.data.replace_data_generator(recipe, data)

Skipped downloading goldenrecipedb with name "xval_det" and variant "Xval0.3" (0), as it already exists.
This is the Cross-validation volume. Available methods are- 
get_golden_df 
describe_table


### Example Dataset: Road Sign Detection (Kaggle)
The steps below will help you retrieve and set up the Road Sign Detection dataset. You can download it directly from [Kaggle](https://www.kaggle.com/datasets/andrewmvd/road-sign-detection).

#### Steps to Use Your Own Dataset:
1. **Download your dataset**:
   - If using the Road Sign Detection dataset from Kaggle, navigate to the dataset page, log in with your Kaggle credentials, and click "Download."
   - Unzip the downloaded file and place the dataset in a local directory.
   
2. **Set the `root_path` for your dataset**:
   - After unzipping, set the `root_path` in your code to point to the folder containing your dataset.
   - Example for the Road Sign Detection dataset:
     ```python
     root_path = "/path/to/road_sign_detection_dataset/"
     ```

3. **Supported Dataset Formats**:
   - LEIP Design supports various formats such as YOLO, COCO, and PASCAL.
   - You can also [integrate datasets from FiftyOne](https://docs.latentai.io/leip/design/latest/content/reference/data_helpers/#leip_recipe_designer.helpers.data.attach_fiftyone_data_generator).

   If your dataset is in any of these formats, you can easily ingest it into LEIP Design using the provided helpers.

4. **Ingest the Dataset into the Recipe**:
   - Once the dataset is prepared, you can attach it to the recipe using our [data ingestion helpers](https://docs.latentai.io/leip/design/latest/content/reference/data_helpers/#format-specific-data-generators):
     ```python
     data = rd.helpers.data.new_pascal_data_generator() # fill based on docs
     rd.helpers.data.replace_data_generator(recipe, data)
     ```

### BYOD Example: Road Sign Detection Dataset
For convenience, if you are using the Road Sign Detection dataset, you can mirror it by running the following command:


In [5]:
# Create a new data generator for the Pascal VOC dataset format - ensure root_path is set
data = rd.helpers.data.new_pascal_data_generator(
    pantry=pantry,
    root_path="${paths.cache_dir}/road-sign-data",
    images_dir="images",
    annotations_dir="annotations",
    nclasses=4,
    is_split=False,
    trainval_split_ratio=0.80,
    trainval_split_seed=42,
    dataset_name="road-sign-data",
    download_url="https://s3.us-west-1.amazonaws.com/leip-showcase.latentai.io/recipes/andrewmvd_road-sign-detection.zip" # skip if pre-downloaded
)

rd.helpers.data.replace_data_generator(recipe, data)

In [ ]:
recipe["data_generator"]

#### Additional Resources:
- [Supported Dataset Formats](https://docs.latentai.io/leip/design/latest/content/reference/data_helpers/#format-specific-data-generators)
- [FiftyOne Integration](https://docs.latentai.io/leip/design/latest/content/reference/data_helpers/#leip_recipe_designer.helpers.data.attach_fiftyone_data_generator)

Once your dataset is loaded, you can proceed to training the recipe just like any other dataset supported in LEIP Design, as shown in the [Getting Started tutorial](https://docs.latentai.io/leip/design/latest/notebooks/GettingStartedwithLEIPDesign/).